# Effect Size for Words

**Computational Text Analysis Pack · Module 04 — Comparing Corpora**

*Technique:* Why significance is not importance; log ratio and %DIFF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/04b_effect_size_for_words.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Log-likelihood has a property that trips up nearly everyone: **it grows with
corpus size**. Double your corpus and every keyness score roughly doubles, even
though nothing about the language changed.

So a big corpus makes trivial differences look overwhelming. The fix is the same
one you met with p-values and Cohen's d in the statistics course: report an
**effect size** alongside the test.

For words, the standard effect size is **log ratio** — how many times more common
a word is in one corpus than the other.

## The tooling

**`collections.Counter`** and plain arithmetic. Keyness needs no library: the
log-likelihood formula is four lines. **quanteda** (R) and **Sketch Engine** are
the field's usual implementations of exactly this.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

In [ ]:
# The demonstration: same language, growing sample, watch keyness inflate.
eng = df[df["language"] == "English"]
a_all = eng[eng["category"] == "politics"]
b_all = eng[eng["category"] != "politics"]

print(f"{'docs used':>10}  {'top keyness':>12}  {'that word':>14}")
print("-" * 42)
for frac in (0.15, 0.3, 0.6, 1.0):
    a = a_all.head(max(5, int(len(a_all) * frac)))
    b = b_all.head(max(5, int(len(b_all) * frac)))
    k = log_likelihood(frequencies(a["text"]), frequencies(b["text"]), min_count=5)
    row = k.iloc[0]
    print(f"{len(a) + len(b):>10}  {row['keyness']:>12.1f}  {row['word']:>14}")
print()
print("Same corpus, same language, same relative frequencies — and keyness")
print("climbs with sample size. It measures evidence, not importance.")

## Log ratio: the effect size

$$\text{log ratio} = \log_2\left(\frac{\text{rate in A}}{\text{rate in B}}\right)$$

Read it directly: **1 means twice as common, 2 means four times, 3 means eight
times.** Unlike keyness it does not change when the corpus grows.

Words absent from one corpus need a small correction, or the ratio is infinite.

In [ ]:
def add_log_ratio(keys, smoothing=0.5):
    k = keys.copy()
    a = k["per_1000_a"].clip(lower=smoothing / 1000)
    b = k["per_1000_b"].clip(lower=smoothing / 1000)
    k["log_ratio"] = np.log2(a / b)
    k["times_more_common"] = 2 ** k["log_ratio"].abs()
    return k

k = add_log_ratio(log_likelihood(frequencies(a_all["text"]),
                                 frequencies(b_all["text"]), min_count=20))

print("Ranked by KEYNESS (statistical evidence):")
print(k.head(10)[["word", "count_a", "per_1000_a", "per_1000_b",
                  "keyness", "log_ratio"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
print("Ranked by LOG RATIO (size of the difference), same threshold:")
big = k[k["count_a"] + k["count_b"] >= 40].sort_values("log_ratio", ascending=False)
print(big.head(10)[["word", "count_a", "count_b", "per_1000_a", "per_1000_b",
                    "keyness", "log_ratio"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("Different lists. The first is dominated by common words with modest")
print("differences; the second by rarer words with dramatic ones. Both are true.")

In [ ]:
# The chart that shows both at once, and is worth putting in a paper.
fig, ax = plt.subplots(figsize=(9, 6))
plot = k[k["count_a"] + k["count_b"] >= 30]
ax.scatter(plot["log_ratio"], plot["keyness"], s=14, alpha=0.35)
for _, r in pd.concat([k.head(8), big.head(8)]).drop_duplicates("word").iterrows():
    ax.annotate(r["word"], (r["log_ratio"], r["keyness"]), fontsize=8,
                xytext=(3, 3), textcoords="offset points")
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("log ratio  (effect size — how many times more common)")
ax.set_ylabel("keyness  (evidence)")
ax.set_title("The words worth writing about are top-RIGHT and top-LEFT")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Find a word high on keyness but near zero on log ratio, and one high on both.

Which would you put in a paper? The second — the first is a common word whose
tiny difference the corpus size made significant.

### 🔧 Your turn 2

Raise the `count_a + count_b` threshold from 30 to 200 and re-run the chart.

Extreme log ratios thin out. Those were words appearing a handful of times, where
the ratio is unstable — a single extra occurrence would move it substantially.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The pattern is consistent: high keyness with near-zero log ratio
is a function word or very common noun whose rate differs by a few percent across
two large corpora. Statistically certain, substantively nothing. High on both is
a word that is genuinely characteristic — that is what belongs in your write-up,
with both numbers quoted.

**Your turn 2.** Yes, and this is the general rule for ratios: they are unstable
when the denominator is small. With 5 occurrences against 1, the ratio is 5 —
and one more occurrence either way changes it dramatically. Setting a minimum
combined count is the same guard as the `min_count` in keyness and the `min_df`
in vectorising, and it is the same Zipf problem underneath (02b).

**How to report a keyness result.** *"'Vaccine' is 6.2 times more common in health
reporting than in other categories (log ratio 2.64; G² = 411.3; 189 vs 34
occurrences)."* Effect size, test statistic, and raw counts. Any of the three
alone can mislead.

</details>

## Where this stops

You can say which words differ and by how much. Next: **04c**, where the corpora
being compared are separated by a century rather than a topic.